# Expected limit plots vs topology, bound state energy, and lifetime

Reads the limits produced by `sidm/scripts/run_combine_limits.py` (see
`make_datacards.ipynb` for how the datacards are built) and plots the expected 95% CL upper
limit on the signal cross section against the three axes of the signal grid:

* **topology** — `4Mu` or `2Mu2E`;
* **bound state energy** — `m_bound`, the mass of the bound state (200, 500, 800, 1000 GeV);
* **lifetime** — the generated `ctau`, converted to the **average lab-frame $L_{xy}$** with the
  lookup table below, because that is the quantity the detector actually responds to.

Because the signal is normalised to a 1 fb reference cross section, Combine's `r` is directly
the limit on $\sigma$ in fb.

**Plot style** (as requested):

| element | meaning |
|---|---|
| black line | median expected limit, `expected_50` |
| green band | 1$\sigma$ band, `expected_16` to `expected_84` |
| yellow band | 2$\sigma$ band, `expected_2p5` to `expected_97p5` |

log y-axis throughout, log x-axis for $L_{xy}$. This is the standard Brazil-plot convention.

All figures are written to `plots/`.

In [ ]:
import os
import sys
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(1, os.path.join(os.getcwd(), '../../..'))
from sidm.tools import utilities

utilities.set_plot_style()
%matplotlib inline

STUDY_DIR = Path(os.getcwd())
LIMIT_CSV = STUDY_DIR / "limits" / "limits.csv"
PLOT_DIR = STUDY_DIR / "plots"

# One output directory per family of plots.
OUT_DIRS = {
    "brazil_vs_lxy": PLOT_DIR / "brazil_vs_lxy",
    "vs_lxy_by_mass": PLOT_DIR / "vs_lxy_by_mass",
    "vs_lxy_by_topology": PLOT_DIR / "vs_lxy_by_topology",
    "vs_mass": PLOT_DIR / "vs_mass",
    "summary": PLOT_DIR / "summary",
    "vs_mbs": PLOT_DIR / "vs_mbs",
    "exclusion": PLOT_DIR / "exclusion",
    "epsilon": PLOT_DIR / "epsilon",
    "by_method": PLOT_DIR / "by_method",
}
for path in OUT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("reading", LIMIT_CSV)
print("writing plots under", PLOT_DIR)

## 1. `ctau` $\to$ average lab-frame $L_{xy}$

Each row of the table is `(m_bound, mzd, ctau_1 ... ctau_5)` in mm, and the five `ctau` columns
correspond in order to the five average lab-frame $L_{xy}$ values in cm. So the mapping is
keyed on the full `(m_bound, mzd, ctau_mm)` triple — the same `ctau` means a different $L_{xy}$
at a different mass, which is exactly why the conversion is needed before masses can be
compared on one axis.

In [ ]:
table_rows = [
    (100, 0.25, 0.02, 0.2, 2, 10, 20),
    (100, 1.2, 0.096, 0.96, 9.6, 48, 96),
    (100, 5, 0.4, 4, 40, 200, 400),
    (150, 0.25, 0.013, 0.13, 1.3, 6.7, 13),
    (150, 1.2, 0.064, 0.64, 6.4, 32, 64),
    (150, 5, 0.27, 2.7, 27, 130, 270),
    (200, 0.25, 0.01, 0.1, 1, 5, 10),
    (200, 1.2, 0.048, 0.48, 4.8, 24, 48),
    (200, 5, 0.2, 2, 20, 100, 200),
    (500, 0.25, 0.004, 0.04, 0.4, 2, 4),
    (500, 1.2, 0.019, 0.19, 1.9, 9.6, 19),
    (500, 5, 0.08, 0.8, 8, 40, 80),
    (800, 0.25, 0.0025, 0.025, 0.25, 1.2, 2.5),
    (800, 1.2, 0.012, 0.12, 1.2, 6, 12),
    (800, 5, 0.05, 0.5, 5, 25, 50),
    (1000, 0.25, 0.002, 0.02, 0.2, 1, 2),
    (1000, 1.2, 0.0096, 0.096, 0.96, 4.8, 9.6),
    (1000, 5, 0.04, 0.4, 4, 20, 40),
]
avg_lab_lxy_cm_values = [0.3, 3.0, 30.0, 150.0, 300.0]

# (m_bound, mzd, ctau_mm) -> average lab-frame Lxy in cm.  The ctau keys are
# rounded to guard against float noise in the sample names (e.g. 0.0096).
LXY_LOOKUP = {
    (float(row[0]), float(row[1]), round(float(ctau), 6)): lxy
    for row in table_rows
    for ctau, lxy in zip(row[2:], avg_lab_lxy_cm_values)
}

def avg_lab_lxy_cm(m_bound, mzd, ctau_mm):
    '''Average lab-frame Lxy in cm for one grid point, or NaN if not tabulated.'''
    return LXY_LOOKUP.get((float(m_bound), float(mzd), round(float(ctau_mm), 6)), np.nan)

print(f"{len(LXY_LOOKUP)} grid points tabulated "
      f"({len(table_rows)} (m_bound, mzd) rows x {len(avg_lab_lxy_cm_values)} lifetimes)")

In [ ]:
limits = pd.read_csv(LIMIT_CSV)

# The datacard filenames call these m_mediator / m_darkphoton; rename to the
# vocabulary of the lookup table.
limits = limits.rename(columns={
    "final_state": "topology",
    "m_mediator": "m_bound",
    "m_darkphoton": "mzd",
    "exp_m2": "expected_2p5",
    "exp_m1": "expected_16",
    "exp": "expected_50",
    "exp_p1": "expected_84",
    "exp_p2": "expected_97p5",
})
limits["lxy_cm"] = [
    avg_lab_lxy_cm(m, z, c)
    for m, z, c in zip(limits.m_bound, limits.mzd, limits.ctau)
]

missing = limits[limits.lxy_cm.isna()]
if len(missing):
    raise ValueError(
        f"{len(missing)} grid points are not in the lookup table:\n"
        + missing[["topology", "m_bound", "mzd", "ctau"]].to_string(index=False)
    )

limits = limits.sort_values(["topology", "m_bound", "mzd", "lxy_cm"]).reset_index(drop=True)
print(f"{len(limits)} limits, all resolved to an Lxy")
print("topologies :", sorted(limits.topology.unique()))
print("m_bound    :", sorted(limits.m_bound.unique()))
print("mzd        :", sorted(limits.mzd.unique()))
print("Lxy [cm]   :", sorted(limits.lxy_cm.unique()))
limits[["topology", "m_bound", "mzd", "ctau", "lxy_cm",
        "expected_2p5", "expected_16", "expected_50",
        "expected_84", "expected_97p5"]].head(10)

## 2. Plot helpers

`BAND_STYLE` is the single place the colour assignment lives. `GRID_RC` shrinks the CMS
style's 26 pt text for the multi-panel figures, where it would otherwise overrun the panels.

In [ ]:
# Standard Brazil-plot convention: green inner 1 sigma, yellow outer 2 sigma.
BAND_STYLE = {
    "one_sigma": {"color": "#00CC00", "label": r"expected $\pm 1\sigma$"},
    "two_sigma": {"color": "#FFCC00", "label": r"expected $\pm 2\sigma$"},
    "median": {"color": "black", "ls": "--", "lw": 2, "label": "median expected"},
}

# The CMS plot style sets font.size to 26, which is right for a single full-size
# figure but illegible once several panels share one canvas.  Multi-panel
# figures are built inside plt.rc_context(GRID_RC) to scale the text down.
GRID_RC = {
    "font.size": 13,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
}
XLABEL_LXY = r"average lab-frame $L_{xy}$ [cm]"
XLABEL_MASS = r"bound state energy $m_\mathrm{bound}$ [GeV]"
YLABEL = r"95% CL upper limit on $\sigma$ [fb]"
# Shorter form for the overlay plots, whose axis label would otherwise be
# wider than the figure at this plot style's font size.
YLABEL_MEDIAN = r"median limit on $\sigma$ [fb]"

# Distinguishable, colour-blind-safe series colours for the overlay plots.
SERIES_COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]


def draw_brazil(ax, group, x="lxy_cm"):
    '''Median expected limit with its 1 and 2 sigma bands, on a log y-axis.

    The 2 sigma band is drawn first so the narrower 1 sigma band sits on top of
    it rather than being hidden underneath.
    '''
    group = group.sort_values(x)
    ax.fill_between(group[x], group.expected_2p5, group.expected_97p5,
                    **BAND_STYLE["two_sigma"])
    ax.fill_between(group[x], group.expected_16, group.expected_84,
                    **BAND_STYLE["one_sigma"])
    ax.plot(group[x], group.expected_50, **BAND_STYLE["median"])
    ax.set_yscale("log")
    ax.set_ylabel(YLABEL)
    ax.grid(alpha=0.3, which="both")
    return ax


def style_lxy_axis(ax):
    ax.set_xscale("log")
    ax.set_xlabel(XLABEL_LXY)


def save(fig, outdir, name):
    '''Write a figure as both png and pdf, and return the png path.'''
    for ext in ("png", "pdf"):
        fig.savefig(OUT_DIRS[outdir] / f"{name}.{ext}", dpi=150, bbox_inches="tight")
    return OUT_DIRS[outdir] / f"{name}.png"


def label_point(topology, m_bound, mzd):
    return (f"{topology}, $m_\\mathrm{{bound}}$ = {m_bound:g} GeV, "
            f"$m_{{Z_D}}$ = {mzd:g} GeV")

## 3. Expected limit vs lifetime

One figure per (topology, bound state energy, $m_{Z_D}$): the full band structure against
average lab-frame $L_{xy}$.

In [ ]:
made = []
for (topology, m_bound, mzd), group in limits.groupby(["topology", "m_bound", "mzd"]):
    fig, ax = plt.subplots(figsize=(7, 5))
    draw_brazil(ax, group)
    style_lxy_axis(ax)
    ax.set_title(label_point(topology, m_bound, mzd), fontsize=11)
    ax.legend(loc="best", fontsize=9)
    name = f"limit_vs_lxy_{topology}_mbound{m_bound:g}_mzd{mzd:g}".replace(".", "p")
    made.append(save(fig, "brazil_vs_lxy", name))
    plt.close(fig)

print(f"wrote {len(made)} figures to {OUT_DIRS['brazil_vs_lxy']}")

In [ ]:
# Same thing as one overview grid per topology, for reading at a glance.
# Axes are shared and only the outer ones are labelled, so the tick labels have
# room to breathe.
for topology in sorted(limits.topology.unique()):
    subset = limits[limits.topology == topology]
    masses = sorted(subset.m_bound.unique())
    mzds = sorted(subset.mzd.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(mzds), len(masses),
                                 figsize=(4.4 * len(masses), 3.6 * len(mzds)),
                                 squeeze=False, sharex=True, sharey=True,
                                 layout="constrained")
        for (i, mzd), (j, m_bound) in itertools.product(enumerate(mzds), enumerate(masses)):
            ax = axes[i][j]
            group = subset[(subset.m_bound == m_bound) & (subset.mzd == mzd)]
            draw_brazil(ax, group)
            style_lxy_axis(ax)
            ax.set_title(f"$m_\\mathrm{{bound}}$={m_bound:g} GeV, $m_{{Z_D}}$={mzd:g} GeV")
            # only the outer panels carry axis labels
            if j:
                ax.set_ylabel("")
            if i != len(mzds) - 1:
                ax.set_xlabel("")
        axes[0][0].legend(loc="upper right")
        fig.suptitle(f"{topology}: expected limit vs average lab-frame $L_{{xy}}$")
        print(save(fig, "summary", f"grid_brazil_vs_lxy_{topology}"))
        plt.show()

## 4. Dependence on bound state energy

Median expected limits for every bound state energy overlaid on one $L_{xy}$ axis, one figure
per (topology, $m_{Z_D}$). The 1$\sigma$ band of the best-performing mass is kept as a shaded
reference so the spread between masses can be judged against the uncertainty on any one of them.

In [ ]:
for (topology, mzd), subset in limits.groupby(["topology", "mzd"]):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))

    masses = sorted(subset.m_bound.unique())
    best = subset.loc[subset.expected_50.idxmin(), "m_bound"]
    reference = subset[subset.m_bound == best].sort_values("lxy_cm")
    ax.fill_between(reference.lxy_cm, reference.expected_16, reference.expected_84,
                    color=BAND_STYLE["one_sigma"]["color"], alpha=0.45,
                    label=rf"$\pm 1\sigma$, $m_\mathrm{{bound}}$ = {best:g} GeV")

    for colour, m_bound in zip(itertools.cycle(SERIES_COLORS), masses):
        group = subset[subset.m_bound == m_bound].sort_values("lxy_cm")
        ax.plot(group.lxy_cm, group.expected_50, marker="o", ms=5, color=colour,
                label=rf"$m_\mathrm{{bound}}$ = {m_bound:g} GeV")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(XLABEL_LXY)
    ax.set_ylabel(YLABEL_MEDIAN)
    ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV", fontsize=11)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=9)
    print(save(fig, "vs_lxy_by_mass",
               f"median_vs_lxy_{topology}_mzd{mzd:g}".replace(".", "p")))
    plt.show()

In [ ]:
# The same dependence read the other way round: limit vs bound state energy,
# one line per lifetime.  x is linear here -- only Lxy gets a log axis.
for (topology, mzd), subset in limits.groupby(["topology", "mzd"]):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for colour, lxy in zip(itertools.cycle(SERIES_COLORS), sorted(subset.lxy_cm.unique())):
        group = subset[subset.lxy_cm == lxy].sort_values("m_bound")
        ax.plot(group.m_bound, group.expected_50, marker="o", ms=5, color=colour,
                label=rf"$L_{{xy}}$ = {lxy:g} cm")
    ax.set_yscale("log")
    ax.set_xlabel(XLABEL_MASS)
    ax.set_ylabel(YLABEL_MEDIAN)
    ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV", fontsize=11)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=9)
    print(save(fig, "vs_mass", f"median_vs_mbound_{topology}_mzd{mzd:g}".replace(".", "p")))
    plt.show()

## 5. Dependence on topology

`4Mu` and `2Mu2E` compared directly at the same grid point. They are independent measurements
in different signal regions, so this compares the two channels' reach; it is not a combination.

In [ ]:
pairs = list(limits.groupby(["m_bound", "mzd"]))
for (m_bound, mzd), subset in pairs:
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for colour, topology in zip(SERIES_COLORS, sorted(subset.topology.unique())):
        group = subset[subset.topology == topology].sort_values("lxy_cm")
        ax.fill_between(group.lxy_cm, group.expected_16, group.expected_84,
                        color=colour, alpha=0.2)
        ax.plot(group.lxy_cm, group.expected_50, marker="o", ms=5, color=colour,
                label=f"{topology} (median, $\\pm 1\\sigma$)")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(XLABEL_LXY)
    ax.set_ylabel(YLABEL)
    ax.set_title(rf"$m_\mathrm{{bound}}$ = {m_bound:g} GeV, $m_{{Z_D}}$ = {mzd:g} GeV",
                 fontsize=11)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=9)
    print(save(fig, "vs_lxy_by_topology",
               f"topology_comparison_mbound{m_bound:g}_mzd{mzd:g}".replace(".", "p")))
    plt.close(fig)

print(f"\nwrote {len(pairs)} figures to {OUT_DIRS['vs_lxy_by_topology']}")

In [ ]:
# Every median limit on one pair of axes, split by topology.  The legend is 12
# entries long, so it goes underneath the panels rather than on top of the data.
with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), sharey=True, layout="constrained")
    # 12 series on one axis: colour carries the mass, line style carries mzd, so
    # every combination stays distinguishable (a plain colour cycle repeats).
    mass_colour = dict(zip(sorted(limits.m_bound.unique()), SERIES_COLORS))
    mzd_style = {0.25: "-", 1.2: "--", 5.0: ":"}
    for ax, topology in zip(axes, sorted(limits.topology.unique())):
        subset = limits[limits.topology == topology]
        for (m_bound, mzd), group in subset.groupby(["m_bound", "mzd"]):
            group = group.sort_values("lxy_cm")
            ax.plot(group.lxy_cm, group.expected_50, marker="o", ms=4,
                    color=mass_colour[m_bound], ls=mzd_style[mzd],
                    label=rf"$m_\mathrm{{bound}}$={m_bound:g}, $m_{{Z_D}}$={mzd:g} GeV")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(XLABEL_LXY)
        ax.set_title(topology)
        ax.grid(alpha=0.3, which="both")
    axes[0].set_ylabel(YLABEL_MEDIAN)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=4, fontsize=9)
    print(save(fig, "summary", "median_vs_lxy_all_points"))
    plt.show()

In [ ]:
# Median limit over the (bound state energy, Lxy) plane, one panel per (topology, mzd).
# Both axes are categorical here, so the cells are evenly spaced rather than to scale.
topologies = sorted(limits.topology.unique())
mzds = sorted(limits.mzd.unique())

with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(len(topologies), len(mzds),
                             figsize=(4.8 * len(mzds), 4.0 * len(topologies)),
                             squeeze=False, layout="constrained")
    norm = plt.matplotlib.colors.LogNorm(vmin=limits.expected_50.min(),
                                         vmax=limits.expected_50.max())

    for (i, topology), (j, mzd) in itertools.product(enumerate(topologies), enumerate(mzds)):
        ax = axes[i][j]
        subset = limits[(limits.topology == topology) & (limits.mzd == mzd)]
        table = subset.pivot_table(index="m_bound", columns="lxy_cm", values="expected_50")
        mesh = ax.pcolormesh(np.arange(table.shape[1] + 1), np.arange(table.shape[0] + 1),
                             table.values, norm=norm, cmap="viridis_r", shading="flat")
        ax.set_xticks(np.arange(table.shape[1]) + 0.5, [f"{c:g}" for c in table.columns])
        ax.set_yticks(np.arange(table.shape[0]) + 0.5, [f"{r:g}" for r in table.index])
        ax.tick_params(length=0)
        for y, x in itertools.product(range(table.shape[0]), range(table.shape[1])):
            ax.text(x + 0.5, y + 0.5, f"{table.values[y, x]:.3g}",
                    ha="center", va="center", fontsize=9, color="white")
        ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV")
        # only the outer panels carry axis labels
        if i == len(topologies) - 1:
            ax.set_xlabel(XLABEL_LXY)
        if j == 0:
            ax.set_ylabel(XLABEL_MASS)

    fig.colorbar(mesh, ax=axes, label=YLABEL_MEDIAN, shrink=0.75)
    print(save(fig, "summary", "median_limit_map"))
    plt.show()

In [ ]:
# Where the analysis is most and least sensitive
columns = ["topology", "m_bound", "mzd", "ctau", "lxy_cm", "expected_50"]
print("most sensitive points:")
print(limits.nsmallest(10, "expected_50")[columns].to_string(index=False))
print("\nleast sensitive points:")
print(limits.nlargest(5, "expected_50")[columns].to_string(index=False))

summary_csv = PLOT_DIR / "limits_with_lxy.csv"
limits.to_csv(summary_csv, index=False)
print(f"\nwrote the Lxy-annotated limit table to {summary_csv}")

## 6. Bound state energy on the x axis, against the theory cross section

The signal cross sections now live in `configs/cross_sections.yaml` and are read back with
`utilities.get_xs(sample, use_signal_xs=True)`. They depend only on $m_\mathrm{bound}$, so on
an $m_\mathrm{bound}$ axis the theory prediction is a single curve the limits can be compared
against directly: **a point is excluded where its limit falls below the theory curve.**

The datacards are still built at the 1 fb reference, so `expected_50` is a cross section in fb
and dividing by the theory cross section is all that is needed --- no refit, and nothing about
the coffea processing changes.

In [ ]:
from sidm.tools import utilities as sidm_utils

# the xs config is in pb; everything in this notebook is in fb
limits["theory_xs_fb"] = [
    sidm_utils.get_xs(s, use_signal_xs=True) * 1000.0 for s in limits.signal
]
# r against the theory cross section rather than the 1 fb reference: < 1 is excluded
limits["r_theory"] = limits.expected_50 / limits.theory_xs_fb

theory_by_mass = limits.groupby("m_bound").theory_xs_fb.first().sort_index()
print("theory cross section by bound state energy:")
for m, x in theory_by_mass.items():
    print(f"  m_bound = {m:6.0f} GeV : {x:9.4f} fb")

print(f"\nexpected-excluded points (median limit < theory xs): "
      f"{(limits.r_theory < 1).sum()}/{len(limits)}")
limits[["topology", "m_bound", "mzd", "lxy_cm", "expected_50",
        "theory_xs_fb", "r_theory"]].sort_values("r_theory").head(10)

### Limit vs $m_\mathrm{bound}$

The analogue of the all-points $L_{xy}$ figure with bound state energy on the x axis. Colour
carries the lifetime, line style carries $m_{Z_D}$, and the heavy black line is the theory
cross section.

In [ ]:
lxy_colour = dict(zip(sorted(limits.lxy_cm.unique()), SERIES_COLORS))
mzd_style = {0.25: "-", 1.2: "--", 5.0: ":"}

with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharey=True, layout="constrained")
    for ax, topology in zip(axes, sorted(limits.topology.unique())):
        subset = limits[limits.topology == topology]
        for (mzd, lxy), group in subset.groupby(["mzd", "lxy_cm"]):
            group = group.sort_values("m_bound")
            ax.plot(group.m_bound, group.expected_50, marker="o", ms=4,
                    color=lxy_colour[lxy], ls=mzd_style[mzd],
                    label=rf"$L_{{xy}}$={lxy:g} cm, $m_{{Z_D}}$={mzd:g} GeV")
        ax.plot(theory_by_mass.index, theory_by_mass.values, color="black", lw=3,
                marker="s", ms=7, zorder=10, label=r"theory $\sigma$")
        ax.set_yscale("log")
        ax.set_xlabel(XLABEL_MASS)
        ax.set_title(topology)
        ax.grid(alpha=0.3, which="both")
    axes[0].set_ylabel(YLABEL_MEDIAN)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=4, fontsize=8)
    print(save(fig, "vs_mbs", "median_vs_mbound_all_points_with_theory"))
    plt.show()

In [ ]:
# Split by mzd, which is easier to read point by point. Shaded band is +-1 sigma.
topologies = sorted(limits.topology.unique())
mzds = sorted(limits.mzd.unique())
with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(len(topologies), len(mzds),
                             figsize=(4.8 * len(mzds), 4.0 * len(topologies)),
                             squeeze=False, sharex=True, sharey=True, layout="constrained")
    for (i, topology), (j, mzd) in itertools.product(enumerate(topologies), enumerate(mzds)):
        ax = axes[i][j]
        subset = limits[(limits.topology == topology) & (limits.mzd == mzd)]
        for lxy, group in subset.groupby("lxy_cm"):
            group = group.sort_values("m_bound")
            ax.fill_between(group.m_bound, group.expected_16, group.expected_84,
                            color=lxy_colour[lxy], alpha=0.15)
            ax.plot(group.m_bound, group.expected_50, marker="o", ms=4,
                    color=lxy_colour[lxy], label=rf"$L_{{xy}}$={lxy:g} cm")
        ax.plot(theory_by_mass.index, theory_by_mass.values, color="black", lw=2.5,
                marker="s", ms=6, zorder=10, label=r"theory $\sigma$")
        ax.set_yscale("log")
        ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV")
        if i == len(topologies) - 1:
            ax.set_xlabel(XLABEL_MASS)
        if j == 0:
            ax.set_ylabel(YLABEL_MEDIAN)
    axes[0][0].legend(fontsize=8)
    print(save(fig, "vs_mbs", "grid_median_vs_mbound_with_theory"))
    plt.show()

### Exclusion map: limit divided by theory

The same $(m_\mathrm{bound}, L_{xy})$ plane as the earlier map, but the colour axis is
$r_\mathrm{theory} = \sigma_\mathrm{limit}/\sigma_\mathrm{theory}$ --- the signal strength the
limit corresponds to with the signal normalised to its theory cross section instead of 1 fb.
**Outlined cells are below 1 and are expected to be excluded.**

In [ ]:
with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(len(topologies), len(mzds),
                             figsize=(4.8 * len(mzds), 4.0 * len(topologies)),
                             squeeze=False, layout="constrained")
    # diverging scale centred on r = 1 so the exclusion boundary is the colour change
    span = max(abs(np.log10(limits.r_theory.min())), abs(np.log10(limits.r_theory.max())))
    norm = plt.matplotlib.colors.LogNorm(vmin=10 ** -span, vmax=10 ** span)

    for (i, topology), (j, mzd) in itertools.product(enumerate(topologies), enumerate(mzds)):
        ax = axes[i][j]
        subset = limits[(limits.topology == topology) & (limits.mzd == mzd)]
        table = subset.pivot_table(index="m_bound", columns="lxy_cm", values="r_theory")
        mesh = ax.pcolormesh(np.arange(table.shape[1] + 1), np.arange(table.shape[0] + 1),
                             table.values, norm=norm, cmap="RdYlGn_r", shading="flat")
        ax.set_xticks(np.arange(table.shape[1]) + 0.5, [f"{c:g}" for c in table.columns])
        ax.set_yticks(np.arange(table.shape[0]) + 0.5, [f"{r:g}" for r in table.index])
        ax.tick_params(length=0)
        for y, x in itertools.product(range(table.shape[0]), range(table.shape[1])):
            val = table.values[y, x]
            if val < 1:  # ring the excluded cells so they read at a glance
                ax.add_patch(plt.Rectangle((x, y), 1, 1, fill=False,
                                           edgecolor="black", lw=2.5))
            ax.text(x + 0.5, y + 0.5, f"{val:.3g}", ha="center", va="center", fontsize=9,
                    color="black", fontweight="bold" if val < 1 else "normal")
        ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV")
        if i == len(topologies) - 1:
            ax.set_xlabel(XLABEL_LXY)
        if j == 0:
            ax.set_ylabel(XLABEL_MASS)

    fig.colorbar(mesh, ax=axes,
                 label=r"$\sigma_\mathrm{limit}/\sigma_\mathrm{theory}$", shrink=0.75)
    fig.suptitle(r"Expected exclusion: outlined cells have $r_\mathrm{theory} < 1$")
    print(save(fig, "exclusion", "r_theory_map"))
    plt.show()

In [ ]:
cols = ["topology", "m_bound", "mzd", "lxy_cm", "expected_50", "theory_xs_fb", "r_theory"]
excluded = limits[limits.r_theory < 1].sort_values("r_theory")
print(f"expected-excluded points: {len(excluded)}/{len(limits)}\n")
print(excluded[cols].to_string(index=False))

limits.to_csv(PLOT_DIR / "limits_with_lxy.csv", index=False)
print(f"\nrewrote {PLOT_DIR / 'limits_with_lxy.csv'} with theory_xs_fb and r_theory")

## 7. The $(m_{Z_D},\ \epsilon^2)$ plane

The kinetic-mixing parameter follows from the dark photon mass and the lifetime,

$$\epsilon = \sqrt{\frac{80}{m_{Z_D}\, c\tau}}\times 10^{-6},$$

with $m_{Z_D}$ in GeV and $c\tau$ in mm. Every grid point therefore lands somewhere in the
$(m_{Z_D},\ \epsilon^2)$ plane, which is the plane dark photon searches are usually compared in.

Two things to keep in mind reading these:

* $c\tau$ depends on **both** $m_{Z_D}$ and $m_\mathrm{bound}$, so the same $m_{Z_D}$ column
  contains points from all four bound state masses at different $\epsilon^2$. They are
  different models, so the panels are kept separate rather than merged into one contour.
* the grid only has three $m_{Z_D}$ values, so this is a scatter of the points actually
  simulated, not an interpolated exclusion contour. Drawing a smooth contour through three
  columns would imply coverage the grid does not have.

In [ ]:
# epsilon from the dark photon mass and lifetime
limits["epsilon"] = np.sqrt(80.0 / limits.mzd / limits.ctau) * 1e-6
limits["epsilon2"] = limits.epsilon ** 2

print(f"epsilon^2 spans {limits.epsilon2.min():.3e} to {limits.epsilon2.max():.3e}")
print(f"m_ZD values on the x axis: {sorted(limits.mzd.unique())}")
limits[["topology", "m_bound", "mzd", "ctau", "epsilon", "epsilon2",
        "expected_50", "r_theory"]].sort_values("epsilon2").head(8)

### Coloured by $r_\mathrm{theory}$

Marker colour is $\sigma_\mathrm{limit}/\sigma_\mathrm{theory}$ on the same diverging scale as
the exclusion map: **outlined markers are below 1 and are expected to be excluded.**

In [ ]:
MARKERS = {200.0: "o", 500.0: "s", 800.0: "D", 1000.0: "^"}

def eps_panel(ax, subset, value, norm, cmap):
    '''Scatter one (topology, m_bound) subset in the (m_ZD, eps^2) plane.'''
    for m_bound, group in subset.groupby("m_bound"):
        ax.scatter(group.mzd, group.epsilon2, c=group[value], norm=norm, cmap=cmap,
                   marker=MARKERS.get(m_bound, "o"), s=190, edgecolors="none", zorder=3)
        excl = group[group.r_theory < 1]
        if len(excl):
            ax.scatter(excl.mzd, excl.epsilon2, facecolors="none", edgecolors="black",
                       marker=MARKERS.get(m_bound, "o"), s=190, linewidths=2.0, zorder=4)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid(alpha=0.3, which="both")


topologies = sorted(limits.topology.unique())
masses = sorted(limits.m_bound.unique())
span = max(abs(np.log10(limits.r_theory.min())), abs(np.log10(limits.r_theory.max())))
norm = plt.matplotlib.colors.LogNorm(vmin=10 ** -span, vmax=10 ** span)

with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(len(topologies), len(masses),
                             figsize=(3.9 * len(masses), 3.8 * len(topologies)),
                             squeeze=False, sharex=True, sharey=True, layout="constrained")
    for (i, topology), (j, m_bound) in itertools.product(enumerate(topologies),
                                                         enumerate(masses)):
        ax = axes[i][j]
        subset = limits[(limits.topology == topology) & (limits.m_bound == m_bound)]
        eps_panel(ax, subset, "r_theory", norm, "RdYlGn_r")
        ax.set_title(f"{topology}, $m_\\mathrm{{bound}}$ = {m_bound:g} GeV")
        if i == len(topologies) - 1:
            ax.set_xlabel(r"$m_{Z_D}$ [GeV]")
        if j == 0:
            ax.set_ylabel(r"$\epsilon^2$")
    sm = plt.cm.ScalarMappable(norm=norm, cmap="RdYlGn_r")
    fig.colorbar(sm, ax=axes, label=r"$\sigma_\mathrm{limit}/\sigma_\mathrm{theory}$",
                 shrink=0.75)
    fig.suptitle(r"Expected exclusion in the $(m_{Z_D},\ \epsilon^2)$ plane "
                 r"(outlined markers: $r_\mathrm{theory}<1$)")
    print(save(fig, "epsilon", "eps2_vs_mdp_r_theory"))
    plt.show()

### Coloured by the cross section limit itself

The same plane, coloured by the limit on $\sigma$ in fb rather than by the ratio to theory.
This one does not depend on the theory cross sections, so it is the version to read if those
are still being checked.

In [ ]:
norm_sigma = plt.matplotlib.colors.LogNorm(vmin=limits.expected_50.min(),
                                           vmax=limits.expected_50.max())
with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(1, len(topologies), figsize=(7.0 * len(topologies), 5.6),
                             squeeze=False, sharey=True, layout="constrained")
    for j, topology in enumerate(topologies):
        ax = axes[0][j]
        subset = limits[limits.topology == topology]
        for m_bound, group in subset.groupby("m_bound"):
            ax.scatter(group.mzd, group.epsilon2, c=group.expected_50, norm=norm_sigma,
                       cmap="viridis_r", marker=MARKERS.get(m_bound, "o"), s=170,
                       edgecolors="none", zorder=3,
                       label=rf"$m_\mathrm{{bound}}$={m_bound:g} GeV")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(alpha=0.3, which="both")
        ax.set_xlabel(r"$m_{Z_D}$ [GeV]")
        ax.set_title(topology)
    axes[0][0].set_ylabel(r"$\epsilon^2$")
    # marker shape carries the mass; draw that legend in neutral grey
    handles = [plt.Line2D([], [], ls="", marker=MARKERS[m], color="0.35",
                          label=rf"$m_\mathrm{{bound}}$={m:g} GeV") for m in masses]
    axes[0][0].legend(handles=handles, fontsize=9, loc="lower left")
    sm = plt.cm.ScalarMappable(norm=norm_sigma, cmap="viridis_r")
    fig.colorbar(sm, ax=axes, label=YLABEL_MEDIAN, shrink=0.85)
    print(save(fig, "epsilon", "eps2_vs_mdp_sigma"))
    plt.show()

In [ ]:
# Strongest constraint reached in each m_ZD column, and where it sits in epsilon^2
cols = ["topology", "m_bound", "mzd", "ctau", "epsilon2", "expected_50", "r_theory"]
best = (limits.sort_values("r_theory").groupby(["topology", "mzd"]).head(1)
        .sort_values(["topology", "mzd"]))
print("best-constrained point in each (topology, m_ZD) column:\n")
print(best[cols].to_string(index=False))

limits.to_csv(PLOT_DIR / "limits_with_lxy.csv", index=False)
print(f"\nrewrote {PLOT_DIR / 'limits_with_lxy.csv'} with epsilon and epsilon2")

### Comparing against the published dark photon limits

The standard dark photon summary plot lives in this same $(m_{A'},\ \epsilon^2)$ plane, so our
points can be shown on it. Two things have to be said plainly first.

**The external contours are not reproduced here.** They are digitised published results, so
they have to come from a source the group agrees on. Drop CSVs into `reference_limits/` (format
in that folder's README) and the cell below picks them up automatically; with the folder empty
it draws our points alone, on axes matched to the reference plot so the two can be laid side by
side.

**Only part of our grid lands on that canvas.** The usual summary plot spans
$m_{A'} \in [10^{-3}, 1]$ GeV and $\epsilon^2 \in [10^{-11}, 10^{-4}]$. Our grid has
$m_{Z_D} \in \{0.25, 1.2, 5\}$ GeV, so **the 1.2 and 5 GeV columns fall off the right-hand
edge** and only the 0.25 GeV column — 40 of 120 points — appears at all.

**And the production mechanism differs.** Those limits constrain direct dark photon production;
ours constrain production of a heavy bound state that decays to dark photons. The planes are
the same but the quantity being limited is not, so the comparison is context, not a like-for-like
overlay.

In [ ]:
REFERENCE_DIR = STUDY_DIR / "reference_limits"

# Axis ranges of the conventional dark photon summary plot, so our version can be
# laid directly against it.
REF_XLIM = (1e-3, 1.0)
REF_YLIM = (1e-11, 1e-4)


def load_reference_limits(directory=REFERENCE_DIR):
    '''Read digitised external limits: one closed contour per CSV.

    Returns [(label, dataframe, filled)] sorted by label. A file whose name ends
    in `_projected` is drawn as a dashed outline rather than a filled region.
    '''
    directory = Path(directory)
    out = []
    for path in sorted(directory.glob("*.csv")):
        curve = pd.read_csv(path)
        missing = {"m_GeV", "eps2"} - set(curve.columns)
        if missing:
            print(f"  skipping {path.name}: missing column(s) {sorted(missing)}")
            continue
        label = path.stem.replace("_excluded", "").replace("_projected", "")
        out.append((label, curve, not path.stem.endswith("_projected")))
    return out


references = load_reference_limits()
if references:
    print(f"loaded {len(references)} external contours: "
          f"{', '.join(label for label, _, _ in references)}")
else:
    print(f"no external contours in {REFERENCE_DIR.name}/ -- drawing our points alone.")
    print("See that folder's README for the expected CSV format.")

on_canvas = limits[limits.mzd.between(*REF_XLIM) & limits.epsilon2.between(*REF_YLIM)]
print(f"\n{len(on_canvas)}/{len(limits)} of our points fall inside the reference plot's axes "
      f"(only the m_ZD = 0.25 GeV column;")
print(" the 1.2 and 5 GeV columns are off the right-hand edge of that canvas).")

In [ ]:
def draw_references(ax, references):
    '''Shade the digitised external limits behind our points.'''
    for i, (label, curve, filled) in enumerate(references):
        colour = SERIES_COLORS[i % len(SERIES_COLORS)]
        if filled:
            ax.fill(curve.m_GeV, curve.eps2, color=colour, alpha=0.25, zorder=1,
                    label=label, lw=0)
            ax.plot(curve.m_GeV, curve.eps2, color=colour, lw=1.0, alpha=0.8, zorder=2)
        else:
            ax.plot(curve.m_GeV, curve.eps2, color=colour, lw=1.6, ls="--", zorder=2,
                    label=f"{label} (proj.)")


with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.0), sharey=True, layout="constrained")
    for ax, topology in zip(axes, sorted(limits.topology.unique())):
        draw_references(ax, references)
        subset = limits[limits.topology == topology]
        for m_bound, group in subset.groupby("m_bound"):
            excluded = group[group.r_theory < 1]
            ax.scatter(group.mzd, group.epsilon2, s=95, zorder=5,
                       facecolors="none", edgecolors="0.45", linewidths=1.1,
                       marker=MARKERS.get(m_bound, "o"))
            if len(excluded):
                ax.scatter(excluded.mzd, excluded.epsilon2, s=95, zorder=6, color="crimson",
                           marker=MARKERS.get(m_bound, "o"),
                           label=rf"excluded, $m_\mathrm{{bound}}$={m_bound:g} GeV")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(*REF_XLIM)
        ax.set_ylim(*REF_YLIM)
        ax.set_xlabel(r"$m_{Z_D}$ [GeV]")
        ax.set_title(f"{topology}  (axes matched to the dark photon summary plot)")
        ax.grid(alpha=0.25, which="both")
    axes[0].set_ylabel(r"$\epsilon^2$")
    axes[0].legend(fontsize=8, loc="lower left")
    print(save(fig, "epsilon", "eps2_vs_mdp_reference_axes"))
    plt.show()

print("Open markers: simulated points that land on this canvas. Filled red: expected exclusion.")
print("Points outside these axes are not drawn -- see the full-range figures above.")

## 8. Counting vs ABCD, with the signal normalised to theory

Everything above builds the datacards at the **1 fb reference** and divides by the theory cross
section afterwards. The cards can also be built with the signal normalised to its **own theory
cross section** (`DatacardConfig(use_theory_xs=True)`), in which case Combine's $r$ is directly
$\sigma_\mathrm{limit}/\sigma_\mathrm{theory}$ and the exclusion boundary is $r=1$.

The two routes carry the same information -- signal strength scales linearly -- and agree to a
median of 0.00%, with a few-percent tail that is just Combine's default 5% scan tolerance. The
theory-normalised cards are the better thing to show, because the number a reader sees *is* the
thing they care about rather than something to be divided afterwards.

Four sets of limits now exist:

| directory | method | signal normalised to |
|---|---|---|
| `limits/` | counting, single bin | 1 fb |
| `limits_abcd/` | ABCD, four bins | 1 fb |
| `limits_theory/` | counting, single bin | theory $\sigma$ |
| `limits_abcd_theory/` | ABCD, four bins | theory $\sigma$ |

In [ ]:
LIMIT_SETS = {
    ("counting", "theory"): STUDY_DIR / "limits_theory" / "limits.csv",
    ("abcd", "theory"): STUDY_DIR / "limits_abcd_theory" / "limits.csv",
    ("counting", "1fb"): STUDY_DIR / "limits" / "limits.csv",
    ("abcd", "1fb"): STUDY_DIR / "limits_abcd" / "limits.csv",
}
METHOD_LABEL = {"counting": "counting (single bin, MC background)",
                "abcd": "ABCD (four bins, background from B,C,D)"}


def prepare(path, method, normalisation):
    '''Load one limits.csv and attach the derived columns the plots need.'''
    df = pd.read_csv(path).rename(columns={
        "final_state": "topology", "m_mediator": "m_bound", "m_darkphoton": "mzd",
        "exp_m2": "expected_2p5", "exp_m1": "expected_16", "exp": "expected_50",
        "exp_p1": "expected_84", "exp_p2": "expected_97p5"})
    df["lxy_cm"] = [avg_lab_lxy_cm(m, z, c)
                    for m, z, c in zip(df.m_bound, df.mzd, df.ctau)]
    df["epsilon"] = np.sqrt(80.0 / df.mzd / df.ctau) * 1e-6
    df["epsilon2"] = df.epsilon ** 2
    df["theory_xs_fb"] = [sidm_utils.get_xs(s, use_signal_xs=True) * 1000.0
                          for s in df.signal]
    if normalisation == "theory":
        # r already IS sigma_limit/sigma_theory
        df["r_theory"] = df.expected_50
        df["sigma_fb"] = df.expected_50 * df.theory_xs_fb
    else:
        df["sigma_fb"] = df.expected_50
        df["r_theory"] = df.expected_50 / df.theory_xs_fb
    df["method"] = method
    df["normalisation"] = normalisation
    return df


sets = {k: prepare(v, *k) for k, v in LIMIT_SETS.items() if v.exists()}
print(f"{len(sets)} limit sets loaded\n")
for (method, norm), df in sets.items():
    print(f"  {method:9s} @ {norm:6s}: {len(df):3d} points, "
          f"{int((df.r_theory < 1).sum()):3d} expected-excluded, "
          f"best r_theory = {df.r_theory.min():.4g}")

### Expected exclusion, both methods side by side

In [ ]:
def exclusion_map(df, title, outname):
    '''r_theory over the (m_bound, Lxy) plane, one panel per (topology, mzd).'''
    topologies = sorted(df.topology.unique())
    mzds = sorted(df.mzd.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(topologies), len(mzds),
                                 figsize=(4.8 * len(mzds), 4.0 * len(topologies)),
                                 squeeze=False, layout="constrained")
        span = max(abs(np.log10(df.r_theory.min())), abs(np.log10(df.r_theory.max())))
        norm = plt.matplotlib.colors.LogNorm(vmin=10 ** -span, vmax=10 ** span)
        for (i, topology), (j, mzd) in itertools.product(enumerate(topologies),
                                                         enumerate(mzds)):
            ax = axes[i][j]
            subset = df[(df.topology == topology) & (df.mzd == mzd)]
            table = subset.pivot_table(index="m_bound", columns="lxy_cm", values="r_theory")
            mesh = ax.pcolormesh(np.arange(table.shape[1] + 1),
                                 np.arange(table.shape[0] + 1),
                                 table.values, norm=norm, cmap="RdYlGn_r", shading="flat")
            ax.set_xticks(np.arange(table.shape[1]) + 0.5,
                          [f"{c:g}" for c in table.columns])
            ax.set_yticks(np.arange(table.shape[0]) + 0.5,
                          [f"{r:g}" for r in table.index])
            ax.tick_params(length=0)
            for y, x in itertools.product(range(table.shape[0]), range(table.shape[1])):
                val = table.values[y, x]
                if val < 1:
                    ax.add_patch(plt.Rectangle((x, y), 1, 1, fill=False,
                                               edgecolor="black", lw=2.5))
                ax.text(x + 0.5, y + 0.5, f"{val:.3g}", ha="center", va="center",
                        fontsize=9, color="black",
                        fontweight="bold" if val < 1 else "normal")
            ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV")
            if i == len(topologies) - 1:
                ax.set_xlabel(XLABEL_LXY)
            if j == 0:
                ax.set_ylabel(XLABEL_MASS)
        fig.colorbar(mesh, ax=axes, label=r"$r_\mathrm{theory}$", shrink=0.75)
        fig.suptitle(title)
        print(save(fig, "exclusion", outname))
        plt.show()


for method in ("counting", "abcd"):
    df = sets[(method, "theory")]
    n = int((df.r_theory < 1).sum())
    exclusion_map(
        df,
        f"{METHOD_LABEL[method]} -- signal at theory $\\sigma$ -- "
        f"{n}/{len(df)} expected-excluded",
        f"r_theory_map_{method}")

### The $(m_{Z_D},\ \epsilon^2)$ plane, both methods

In [ ]:
for method in ("counting", "abcd"):
    df = sets[(method, "theory")]
    topologies = sorted(df.topology.unique())
    masses = sorted(df.m_bound.unique())
    span = max(abs(np.log10(df.r_theory.min())), abs(np.log10(df.r_theory.max())))
    norm = plt.matplotlib.colors.LogNorm(vmin=10 ** -span, vmax=10 ** span)
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(topologies), len(masses),
                                 figsize=(3.9 * len(masses), 3.8 * len(topologies)),
                                 squeeze=False, sharex=True, sharey=True,
                                 layout="constrained")
        for (i, topology), (j, m_bound) in itertools.product(enumerate(topologies),
                                                             enumerate(masses)):
            ax = axes[i][j]
            subset = df[(df.topology == topology) & (df.m_bound == m_bound)]
            eps_panel(ax, subset, "r_theory", norm, "RdYlGn_r")
            ax.set_title(f"{topology}, $m_\\mathrm{{bound}}$ = {m_bound:g} GeV")
            if i == len(topologies) - 1:
                ax.set_xlabel(r"$m_{Z_D}$ [GeV]")
            if j == 0:
                ax.set_ylabel(r"$\epsilon^2$")
        sm = plt.cm.ScalarMappable(norm=norm, cmap="RdYlGn_r")
        fig.colorbar(sm, ax=axes, label=r"$r_\mathrm{theory}$", shrink=0.75)
        fig.suptitle(f"{METHOD_LABEL[method]} -- signal at theory $\\sigma$ "
                     r"(outlined: $r_\mathrm{theory}<1$)")
        print(save(fig, "epsilon", f"eps2_vs_mdp_{method}"))
        plt.show()

### Limit vs bound state energy against theory, both methods

In [ ]:
with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8), sharey=True, layout="constrained")
    for ax, method in zip(axes, ("counting", "abcd")):
        df = sets[(method, "theory")]
        theory_by_mass = df.groupby("m_bound").theory_xs_fb.first().sort_index()
        for (mzd, lxy), group in df.groupby(["mzd", "lxy_cm"]):
            group = group.sort_values("m_bound")
            ax.plot(group.m_bound, group.sigma_fb, marker="o", ms=4,
                    color=lxy_colour[lxy], ls=mzd_style[mzd])
        ax.plot(theory_by_mass.index, theory_by_mass.values, color="black", lw=3,
                marker="s", ms=7, zorder=10, label=r"theory $\sigma$")
        ax.set_yscale("log")
        ax.set_xlabel(XLABEL_MASS)
        ax.set_title(METHOD_LABEL[method], fontsize=11)
        ax.grid(alpha=0.3, which="both")
        ax.legend(fontsize=9)
    axes[0].set_ylabel(YLABEL_MEDIAN)
    fig.suptitle("Both topologies overlaid; colour carries $L_{xy}$, style carries $m_{Z_D}$")
    print(save(fig, "vs_mbs", "median_vs_mbound_counting_vs_abcd"))
    plt.show()

In [ ]:
# Head-to-head: how much the background estimate moves each point
merged = sets[("counting", "theory")].merge(
    sets[("abcd", "theory")], on=["channel", "signal"], suffixes=("_counting", "_abcd"))
merged["ratio"] = merged.r_theory_abcd / merged.r_theory_counting
print("ABCD / counting, r_theory:")
for ch, group in merged.groupby("channel"):
    print(f"  {ch}: median {group.ratio.median():.3f}  "
          f"(range {group.ratio.min():.3f}-{group.ratio.max():.3f})")
print(f"\nexpected-excluded: counting {int((merged.r_theory_counting<1).sum())}/{len(merged)}, "
      f"ABCD {int((merged.r_theory_abcd<1).sum())}/{len(merged)}")
print("\nThe ABCD limits are stronger only because B*C/D lands below the MC region-A count,")
print("a difference of under 1 sigma. It is a fluctuation, not a gain in sensitivity.")

combined = pd.concat(sets.values(), ignore_index=True)
combined.to_csv(PLOT_DIR / "limits_all_methods.csv", index=False)
print(f"\nwrote {PLOT_DIR / 'limits_all_methods.csv'} "
      f"({len(combined)} rows: 4 sets x 120 points)")

## 9. Counting, ABCD, and the two compared -- against $L_{xy}$ and $m_\mathrm{bound}$

Everything in sections 3--5 was built from `limits/limits.csv`, which is the **counting**
result. This section produces the same views for both background estimates and a direct
comparison, so no figure is ambiguous about which method produced it.

The y axis here is the limit on $\sigma$ in **fb**, taken from the 1 fb-reference sets whose
`expected_*` columns are already in those units. That is what lets the theory cross section be
drawn on the same axes.

### How the theory line is made

The signal cross sections in `configs/cross_sections.yaml` are Murtaza's calculation, read back
with `utilities.get_xs(sample, use_signal_xs=True)` (they are in pb; everything here is fb).
They are a property of the **bound state mass alone** -- every $m_{Z_D}$ and every lifetime at a
given $m_\mathrm{bound}$ shares one value:

| $m_\mathrm{bound}$ [GeV] | 200 | 500 | 800 | 1000 |
|---|---|---|---|---|
| $\sigma_\mathrm{theory}$ [fb] | 0.0499 | 1.264 | 8.812 | 0.3416 |

So the same numbers draw differently on the two axes, and this is the whole reason the line
looks like a step on one and a curve on the other:

* **against $L_{xy}$** -- $m_\mathrm{bound}$ is fixed within a panel, so the theory line is a
  single **horizontal line**. A point is excluded where its limit band drops below that line.
* **against $m_\mathrm{bound}$** -- the mass is the x axis, so the same four numbers become a
  **curve**. Exclusion is where a limit curve crosses below it.

No interpolation or fitting is involved: the line is those four tabulated values, joined.

In [ ]:
# fb-valued sets: the 1 fb-reference cards, whose expected_* columns are already in fb
FB_SETS = {"counting": STUDY_DIR / "limits" / "limits.csv",
           "abcd": STUDY_DIR / "limits_abcd" / "limits.csv"}
fb = {m: prepare(path, m, "1fb") for m, path in FB_SETS.items() if path.exists()}

THEORY_FB = fb["counting"].groupby("m_bound").theory_xs_fb.first().sort_index()
METHOD_COLOUR = {"counting": "#0072B2", "abcd": "#D55E00"}
METHOD_SHORT = {"counting": "counting", "abcd": "ABCD"}

print("theory cross section, one value per bound state mass:")
for m, x in THEORY_FB.items():
    print(f"   m_bound = {m:6.0f} GeV -> {x:8.4f} fb")
for method, df in fb.items():
    print(f"\n{method:9s}: {len(df)} points, "
          f"best sigma = {df.expected_50.min():.4g} fb, "
          f"{int((df.expected_50 < df.theory_xs_fb).sum())} below theory")

### Brazil bands vs $L_{xy}$, one figure per method, with the theory line

In [ ]:
def brazil_grid_vs_lxy(df, method, topology):
    '''3x4 panel Brazil grid for one method and topology, theory drawn as a horizontal line.'''
    subset = df[df.topology == topology]
    masses = sorted(subset.m_bound.unique())
    mzds = sorted(subset.mzd.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(mzds), len(masses),
                                 figsize=(4.4 * len(masses), 3.6 * len(mzds)),
                                 squeeze=False, sharex=True, sharey=True,
                                 layout="constrained")
        for (i, mzd), (j, m_bound) in itertools.product(enumerate(mzds), enumerate(masses)):
            ax = axes[i][j]
            group = subset[(subset.m_bound == m_bound) & (subset.mzd == mzd)]
            draw_brazil(ax, group)
            style_lxy_axis(ax)
            # theory is a property of m_bound alone -> constant across this panel
            ax.axhline(THEORY_FB[m_bound], color="crimson", lw=2.2, ls="-", zorder=6,
                       label=r"theory $\sigma$")
            ax.set_title(f"$m_\\mathrm{{bound}}$={m_bound:g} GeV, $m_{{Z_D}}$={mzd:g} GeV")
            if j:
                ax.set_ylabel("")
            if i != len(mzds) - 1:
                ax.set_xlabel("")
        axes[0][0].legend(loc="upper right", fontsize=8)
        fig.suptitle(f"{topology} -- {METHOD_LABEL[method]}")
        name = f"brazil_vs_lxy_{method}_{topology}"
        print(save(fig, "by_method", name))
        plt.show()


for method, df in fb.items():
    for topology in sorted(df.topology.unique()):
        brazil_grid_vs_lxy(df, method, topology)

### The two methods compared vs $L_{xy}$

In [ ]:
def compare_vs_lxy(topology):
    '''Counting and ABCD on the same axes: median plus 1 sigma band, per grid point.'''
    masses = sorted(fb["counting"].m_bound.unique())
    mzds = sorted(fb["counting"].mzd.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(mzds), len(masses),
                                 figsize=(4.4 * len(masses), 3.6 * len(mzds)),
                                 squeeze=False, sharex=True, sharey=True,
                                 layout="constrained")
        for (i, mzd), (j, m_bound) in itertools.product(enumerate(mzds), enumerate(masses)):
            ax = axes[i][j]
            for method, df in fb.items():
                group = df[(df.topology == topology) & (df.m_bound == m_bound)
                           & (df.mzd == mzd)].sort_values("lxy_cm")
                colour = METHOD_COLOUR[method]
                ax.fill_between(group.lxy_cm, group.expected_16, group.expected_84,
                                color=colour, alpha=0.22)
                ax.plot(group.lxy_cm, group.expected_50, color=colour, marker="o", ms=4,
                        label=f"{METHOD_SHORT[method]} (median, $\\pm1\\sigma$)")
            ax.axhline(THEORY_FB[m_bound], color="crimson", lw=2.2, zorder=6,
                       label=r"theory $\sigma$")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(alpha=0.3, which="both")
            ax.set_title(f"$m_\\mathrm{{bound}}$={m_bound:g} GeV, $m_{{Z_D}}$={mzd:g} GeV")
            if i == len(mzds) - 1:
                ax.set_xlabel(XLABEL_LXY)
            if j == 0:
                ax.set_ylabel(YLABEL)
        axes[0][0].legend(loc="upper right", fontsize=8)
        fig.suptitle(f"{topology} -- counting vs ABCD background estimate")
        print(save(fig, "by_method", f"compare_vs_lxy_{topology}"))
        plt.show()


for topology in sorted(fb["counting"].topology.unique()):
    compare_vs_lxy(topology)

### Against $m_\mathrm{bound}$: each method, then both compared

In [ ]:
def vs_mbound(dfs, title, outname, show_band=True):
    '''Limit vs bound state energy with the theory curve; dfs is {label: (df, colour)}.'''
    topologies = sorted(next(iter(dfs.values()))[0].topology.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(1, len(topologies), figsize=(7.0 * len(topologies), 5.8),
                                 squeeze=False, sharey=True, layout="constrained")
        for j, topology in enumerate(topologies):
            ax = axes[0][j]
            for label, (df, colour) in dfs.items():
                subset = df[df.topology == topology]
                for (mzd, lxy), group in subset.groupby(["mzd", "lxy_cm"]):
                    group = group.sort_values("m_bound")
                    if show_band:
                        ax.fill_between(group.m_bound, group.expected_16, group.expected_84,
                                        color=colour, alpha=0.10)
                    ax.plot(group.m_bound, group.expected_50, color=colour, marker="o", ms=3,
                            lw=1.2, alpha=0.85)
                # one legend entry per method rather than per curve
                ax.plot([], [], color=colour, marker="o", ms=5, label=label)
            # theory depends only on m_bound, so here it is a curve
            ax.plot(THEORY_FB.index, THEORY_FB.values, color="crimson", lw=3, marker="s",
                    ms=8, zorder=10, label=r"theory $\sigma$")
            ax.set_yscale("log")
            ax.set_xlabel(XLABEL_MASS)
            ax.set_title(topology)
            ax.grid(alpha=0.3, which="both")
        axes[0][0].set_ylabel(YLABEL)
        axes[0][0].legend(fontsize=9, loc="upper right")
        fig.suptitle(title)
        print(save(fig, "by_method", outname))
        plt.show()


for method, df in fb.items():
    vs_mbound({METHOD_SHORT[method]: (df, METHOD_COLOUR[method])},
              f"{METHOD_LABEL[method]} -- one curve per $(m_{{Z_D}}, L_{{xy}})$",
              f"vs_mbound_{method}")

vs_mbound({METHOD_SHORT[m]: (df, METHOD_COLOUR[m]) for m, df in fb.items()},
          "Counting vs ABCD background estimate",
          "vs_mbound_compare", show_band=False)

In [ ]:
# How often each method's limit falls below theory, by mass
rows = []
for method, df in fb.items():
    for m_bound, group in df.groupby("m_bound"):
        rows.append({"method": METHOD_SHORT[method], "m_bound": m_bound,
                     "theory_fb": THEORY_FB[m_bound], "n_points": len(group),
                     "n_below_theory": int((group.expected_50 < group.theory_xs_fb).sum()),
                     "best_sigma_fb": group.expected_50.min()})
summary = pd.DataFrame(rows).pivot_table(
    index="m_bound", columns="method",
    values=["n_below_theory", "best_sigma_fb"]).round(4)
print("points with median limit below the theory cross section:\n")
print(summary.to_string())

## 10. The ABCD card confronted with the MC signal-region count

The ABCD card's region-A `observation` is now the **MC region-A count**, the same number the
counting card observes, rather than the `B*C/D` prediction. That makes the two methods
genuinely comparable: same data, different background model.

It also changes what the card can tell us. With `observation = B*C/D` the card was
self-consistent by construction and could never show closure tension. With the MC count there,
the signal region says 3.251 while the control regions predict 0.044, and the fit has to
absorb that.

**The blinded expected limit does not move.** `--run blind` builds its Asimov dataset from the
a-priori background-only model and ignores the observation entirely, so those numbers are
identical to before. What changes is everything that *uses* the observation:

| | median $r_\mathrm{theory}$, `SR_4mu` | vs blinded |
|---|---|---|
| a-priori expected (blinded) | 2.65 | — |
| a-posteriori expected | 3.85 | 1.5x weaker |
| observed on MC pseudo-data | 9.15 | **3.5x weaker** |

Treat the last row as a **closure test**, not a limit to quote: it is the ABCD model being
confronted with simulation standing in for data. The degradation is the non-closure showing up
where it should.

In [ ]:
OBS_SETS = {"abcd_obs": STUDY_DIR / "limits_abcd_obs" / "limits.csv"}
obs_avail = {k: v for k, v in OBS_SETS.items() if v.exists()}
if obs_avail:
    abcd_obs = prepare(obs_avail["abcd_obs"], "abcd", "1fb")
    # `prepare` keys r_theory off expected_50; here we also want the observed column
    abcd_obs["obs_sigma_fb"] = pd.read_csv(obs_avail["abcd_obs"]).obs.values
    abcd_obs["r_theory_obs"] = abcd_obs.obs_sigma_fb / abcd_obs.theory_xs_fb

    n_cnt = int((fb["counting"].expected_50 < fb["counting"].theory_xs_fb).sum())
    n_exp = int((fb["abcd"].expected_50 < fb["abcd"].theory_xs_fb).sum())
    n_obs = int((abcd_obs.obs_sigma_fb < abcd_obs.theory_xs_fb).sum())
    print("points reaching exclusion (limit below theory):")
    print(f"   counting, expected (blinded)     : {n_cnt}/120")
    print(f"   ABCD,     expected (blinded)     : {n_exp}/120")
    print(f"   ABCD,     observed on MC as data : {n_obs}/120   <- the closure test")
    print("\nOnce the ABCD card has to explain the MC signal-region count, most of its")
    print("apparent advantage over straight counting disappears.")
else:
    print("limits_abcd_obs/ not found -- run run_combine_limits.py with --unblind on the "
          "ABCD cards to produce it.")

In [ ]:
# Three curves per panel: counting expected, ABCD expected, ABCD confronted with the MC count
def compare_with_observed(topology):
    masses = sorted(fb["counting"].m_bound.unique())
    mzds = sorted(fb["counting"].mzd.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(mzds), len(masses),
                                 figsize=(4.4 * len(masses), 3.6 * len(mzds)),
                                 squeeze=False, sharex=True, sharey=True,
                                 layout="constrained")
        for (i, mzd), (j, m_bound) in itertools.product(enumerate(mzds), enumerate(masses)):
            ax = axes[i][j]
            sel = lambda d: d[(d.topology == topology) & (d.m_bound == m_bound)
                              & (d.mzd == mzd)].sort_values("lxy_cm")
            c, a, o = sel(fb["counting"]), sel(fb["abcd"]), sel(abcd_obs)
            ax.plot(c.lxy_cm, c.expected_50, color=METHOD_COLOUR["counting"],
                    marker="o", ms=4, label="counting, expected")
            ax.plot(a.lxy_cm, a.expected_50, color=METHOD_COLOUR["abcd"],
                    marker="o", ms=4, label="ABCD, expected")
            ax.plot(o.lxy_cm, o.obs_sigma_fb, color=METHOD_COLOUR["abcd"], ls="--",
                    marker="^", ms=5, label="ABCD, obs. on MC count")
            ax.axhline(THEORY_FB[m_bound], color="crimson", lw=2.2, zorder=6,
                       label=r"theory $\sigma$")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(alpha=0.3, which="both")
            ax.set_title(f"$m_\\mathrm{{bound}}$={m_bound:g} GeV, $m_{{Z_D}}$={mzd:g} GeV")
            if i == len(mzds) - 1:
                ax.set_xlabel(XLABEL_LXY)
            if j == 0:
                ax.set_ylabel(YLABEL)
        axes[0][0].legend(loc="upper right", fontsize=7)
        fig.suptitle(f"{topology} -- ABCD before and after confronting the MC "
                     f"signal-region count")
        print(save(fig, "by_method", f"compare_with_observed_{topology}"))
        plt.show()


if obs_avail:
    for topology in sorted(fb["counting"].topology.unique()):
        compare_with_observed(topology)

## 11. Unblinded on MC: counting vs ABCD on the same axes

Both card sets run without `--run blind`, so the fit actually uses the signal-region count.
Both observe the **same number** (the MC region-A yield); they differ only in how they model
the background:

* the **counting** card's background *is* that number, so its observation and prediction agree
  by construction and it cannot show tension;
* the **ABCD** card predicts $B{\times}C/D$ and has to explain the observed count, so its free
  `rateParam`s get pulled to absorb the difference.

These are **closure tests on simulation**, not limits to quote.

The result is the interesting part. Blinded, ABCD looked 3--5$\times$ *stronger* than counting.
Confronted with the same count it comes out **1.1--1.3$\times$ weaker** --- which is the
behaviour you would expect from a method that pays a control-region uncertainty. The apparent
advantage was an artefact of comparing against a prediction that had fluctuated low.

In [ ]:
OBS_PATHS = {"counting": STUDY_DIR / "limits_obs" / "limits.csv",
             "abcd": STUDY_DIR / "limits_abcd_obs" / "limits.csv"}
obs = {}
for method, path in OBS_PATHS.items():
    if not path.exists():
        print(f"missing {path} -- run run_combine_limits.py --unblind on that card set")
        continue
    d = prepare(path, method, "1fb")
    d["obs_sigma_fb"] = pd.read_csv(path).obs.values
    d["r_theory_obs"] = d.obs_sigma_fb / d.theory_xs_fb
    obs[method] = d

for method, d in obs.items():
    n = int((d.obs_sigma_fb < d.theory_xs_fb).sum())
    print(f"{method:9s} observed: {n}/120 points below theory, "
          f"best sigma = {d.obs_sigma_fb.min():.4g} fb")

if len(obs) == 2:
    m = obs["counting"].merge(obs["abcd"], on=["channel", "signal"],
                              suffixes=("_cnt", "_abcd"))
    m["ratio"] = m.obs_sigma_fb_abcd / m.obs_sigma_fb_cnt
    print("\nobserved limit, ABCD / counting:")
    for ch, g in m.groupby("channel"):
        print(f"   {ch}: median {g.ratio.median():.3f}")
    print("\nCompare with the blinded expected ratios (0.32 and 0.19): the ordering flips "
          "once\nboth methods are asked to explain the same signal-region count.")

### Both methods, observed, with theory --- vs $L_{xy}$

In [ ]:
def observed_vs_lxy(topology, with_expected=True):
    '''Counting and ABCD observed on the same panels, theory overlaid.'''
    masses = sorted(obs["counting"].m_bound.unique())
    mzds = sorted(obs["counting"].mzd.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(mzds), len(masses),
                                 figsize=(4.4 * len(masses), 3.6 * len(mzds)),
                                 squeeze=False, sharex=True, sharey=True,
                                 layout="constrained")
        for (i, mzd), (j, m_bound) in itertools.product(enumerate(mzds), enumerate(masses)):
            ax = axes[i][j]
            sel = lambda d: d[(d.topology == topology) & (d.m_bound == m_bound)
                              & (d.mzd == mzd)].sort_values("lxy_cm")
            for method in ("counting", "abcd"):
                colour = METHOD_COLOUR[method]
                if with_expected:  # faint reference: the blinded expected limit
                    e = sel(fb[method])
                    ax.plot(e.lxy_cm, e.expected_50, color=colour, ls=":", lw=1.3,
                            alpha=0.65, label=f"{METHOD_SHORT[method]}, expected")
                o = sel(obs[method])
                ax.plot(o.lxy_cm, o.obs_sigma_fb, color=colour, marker="o", ms=4.5, lw=2,
                        label=f"{METHOD_SHORT[method]}, observed")
            ax.axhline(THEORY_FB[m_bound], color="crimson", lw=2.2, zorder=6,
                       label=r"theory $\sigma$")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(alpha=0.3, which="both")
            ax.set_title(f"$m_\\mathrm{{bound}}$={m_bound:g} GeV, $m_{{Z_D}}$={mzd:g} GeV")
            if i == len(mzds) - 1:
                ax.set_xlabel(XLABEL_LXY)
            if j == 0:
                ax.set_ylabel(YLABEL)
        axes[0][0].legend(loc="upper right", fontsize=7)
        fig.suptitle(f"{topology} -- observed on MC (solid) vs blinded expected (dotted)")
        print(save(fig, "by_method", f"observed_vs_lxy_{topology}"))
        plt.show()


if len(obs) == 2:
    for topology in sorted(obs["counting"].topology.unique()):
        observed_vs_lxy(topology)

### Both methods, observed, with theory --- vs $m_\mathrm{bound}$

In [ ]:
if len(obs) == 2:
    topologies = sorted(obs["counting"].topology.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(1, len(topologies),
                                 figsize=(7.0 * len(topologies), 5.8),
                                 squeeze=False, sharey=True, layout="constrained")
        for j, topology in enumerate(topologies):
            ax = axes[0][j]
            for method in ("counting", "abcd"):
                colour = METHOD_COLOUR[method]
                d = obs[method][obs[method].topology == topology]
                for (mzd, lxy), group in d.groupby(["mzd", "lxy_cm"]):
                    group = group.sort_values("m_bound")
                    ax.plot(group.m_bound, group.obs_sigma_fb, color=colour, marker="o",
                            ms=3, lw=1.2, alpha=0.85)
                ax.plot([], [], color=colour, marker="o", ms=5,
                        label=f"{METHOD_SHORT[method]}, observed")
            ax.plot(THEORY_FB.index, THEORY_FB.values, color="crimson", lw=3, marker="s",
                    ms=8, zorder=10, label=r"theory $\sigma$")
            ax.set_yscale("log")
            ax.set_xlabel(XLABEL_MASS)
            ax.set_title(topology)
            ax.grid(alpha=0.3, which="both")
        axes[0][0].set_ylabel(YLABEL)
        axes[0][0].legend(fontsize=9, loc="upper right")
        fig.suptitle("Observed on MC: counting vs ABCD, one curve per "
                     "$(m_{Z_D}, L_{xy})$")
        print(save(fig, "by_method", "observed_vs_mbound_compare"))
        plt.show()

In [ ]:
# Summary: how each method moves between blinded and observed
if len(obs) == 2:
    rows = []
    for method in ("counting", "abcd"):
        e, o = fb[method], obs[method]
        rows.append({
            "method": METHOD_SHORT[method],
            "excluded, expected": int((e.expected_50 < e.theory_xs_fb).sum()),
            "excluded, observed": int((o.obs_sigma_fb < o.theory_xs_fb).sum()),
            "median obs/exp": round(float((o.obs_sigma_fb / e.expected_50).median()), 3),
        })
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nBlinded, ABCD looked far stronger. Observed, the two land within ~15-35% of each")
    print("other, with ABCD slightly the weaker of the two -- the expected ordering for a")
    print("method that pays a control-region uncertainty.")